# LLM Workshop Code Along

This notebook mirrors `presentation.qmd` and gives runnable demos for each section.

## Agenda

1. LLM Foundations\n
2. RAG\n
3. LLM Evals\n
4. LLM Agents\n
5. MCP and App Patterns

In [2]:
import os
import re
import json
import math
import random
import sqlite3
import statistics
from collections import Counter
import tiktoken
import numpy as np
import pandas as pd

random.seed(42)

## LLM Foundations: Tokenization

In [15]:
enc = tiktoken.encoding_for_model("gpt-2")

sample = "NotebookLM is a RAG app for source-grounded Q&A."
tokens = enc.encode(sample)

print(tokens)
print("Token count:", len(tokens))
print("Decoded tokens:", [enc.decode([t]) for t in tokens])

[6425, 2070, 31288, 318, 257, 371, 4760, 598, 329, 2723, 12, 2833, 276, 1195, 5, 32, 13]
Token count: 17
Decoded tokens: ['Note', 'book', 'LM', ' is', ' a', ' R', 'AG', ' app', ' for', ' source', '-', 'ground', 'ed', ' Q', '&', 'A', '.']


## LLM Foundations: Vector embeddings

In [8]:
# !pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [17]:
embeddings = model.encode("Miami Universoty of Ohio")
print(embeddings[:10])
print(embeddings.shape)  # (2, 384)

[ 0.02065466 -0.03984684 -0.02121316  0.0223866   0.01308369 -0.03353535
 -0.03918867 -0.0909925  -0.03492295  0.04086898]
(384,)


### Euclidean Distance

$$
d(\mathbf{a}, \mathbf{b}) = \|\mathbf{a} - \mathbf{b}\| = \sqrt{\sum_{i=1}^{n}(a_i - b_i)^2}
$$

### Cosine Similarity

$$
\text{cos}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|} = \frac{\sum_{i=1}^{n} a_i b_i}{\sqrt{\sum_{i=1}^{n} a_i^2} \; \sqrt{\sum_{i=1}^{n} b_i^2}}
$$

### Comparison

| | Euclidean Distance | Cosine Similarity |
|---|---|---|
| **Measures** | Magnitude of separation in space | Angular similarity of direction |
| **Range** | $[0, \infty)$ | $[-1, 1]$ |
| **Identical vectors** | $0$ | $1$ |
| **Orthogonal vectors** | $\sqrt{\|\mathbf{a}\|^2 + \|\mathbf{b}\|^2}$ | $0$ |
| **Scale-invariant** | No | Yes |
| **Best for** | Clustering, k-NN | Retrieval, semantic search |



> cosine gives more spread between similar items, while Euclidean gives more spread between dissimilar items. 





In [18]:
def euclidean_distance(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [19]:
texts = [
    "emergency stop procedure for CNC lathe",
    "how to halt the CNC machine in an emergency",
    "the weather in Ohio today",
]

embs = model.encode(texts)

for i in range(len(texts)):
    for j in range(i + 1, len(texts)):
        print(f"\n'{texts[i]}' vs '{texts[j]}'")
        print(f"  Cosine Similarity:   {cosine_similarity(embs[i], embs[j]):.4f}")
        print(f"  Euclidean Distance:  {euclidean_distance(embs[i], embs[j]):.4f}")


'emergency stop procedure for CNC lathe' vs 'how to halt the CNC machine in an emergency'
  Cosine Similarity:   0.5908
  Euclidean Distance:  0.9047

'emergency stop procedure for CNC lathe' vs 'the weather in Ohio today'
  Cosine Similarity:   -0.0166
  Euclidean Distance:  1.4259

'how to halt the CNC machine in an emergency' vs 'the weather in Ohio today'
  Cosine Similarity:   0.0377
  Euclidean Distance:  1.3873


## LLM Foundations: `temperature`

Higher temp = more random

In [30]:
logits = np.array([2.0, 1.0, 0.5])
labels = ["A", "B", "C"]

def softmax(logits):
    scaled = logits 
    exp = np.exp(scaled - scaled.max())
    return exp / exp.sum()

temps = [0.1, 0.5, 1.0, 2.0]

print(softmax(logits/1)) # raw probabilits  
print(softmax(logits/0.3)) # low temp -> more determinisitc/less random
print(softmax(logits/10)) # high temp -> less determinisitc/more random

[0.62853172 0.2312239  0.14024438]
[0.95931365 0.03422255 0.0064638 ]
[0.36159233 0.32718227 0.3112254 ]


# Open AI model

In [1]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print('Set OPENAI_API_KEY to run a live model call.')
else:
    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        response = client.responses.create(
            model='gpt-4.1',
            input='Summarize why grounding matters in RAG in 2 bullets.',
            temperature=0,
            max_output_tokens=120,
            reasoning={'effort': 'low'}
        )
        print(response.output_text)
    except Exception as e:
        print('Live call skipped:', e)

NameError: name 'os' is not defined

## RAG: SQL + Vector Hybrid Retrieval

In [ ]:
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.execute('CREATE TABLE policy_updates (quarter TEXT, topic TEXT, summary TEXT)')
cur.executemany(
    'INSERT INTO policy_updates VALUES (?, ?, ?)',
    [
        ('2025Q4', 'security', 'Added stricter data retention controls.'),
        ('2025Q4', 'privacy', 'Expanded PII masking in logs.'),
        ('2026Q1', 'quality', 'Introduced continuous eval gating in CI.')
    ]
)
conn.commit()

def retrieve_sql(quarter):
    # Parameterized query protects against SQL injection.
    cur.execute('SELECT topic, summary FROM policy_updates WHERE quarter = ?', (quarter,))
    return cur.fetchall()

def retrieve_vector(question, k=2):
    qv = embed(question)
    ranked = sorted([(cosine(qv, dv), d) for d, dv in zip(docs, doc_vectors)], reverse=True)
    return [d for _, d in ranked[:k]]

print('SQL results:', retrieve_sql('2025Q4'))
print('Vector results:', retrieve_vector('How do we handle semantic retrieval?'))

In [ ]:
def build_rag_context(question, quarter='2025Q4'):
    sql_hits = retrieve_sql(quarter)
    vec_hits = retrieve_vector(question, k=2)

    sql_text = '\n'.join([f'- {topic}: {summary}' for topic, summary in sql_hits])
    vec_text = '\n'.join([f'- {v}' for v in vec_hits])

    context = f"Structured facts:\n{sql_text}\n\nSemantic context:\n{vec_text}"
    return context

question = 'What changed recently and why does retrieval quality matter?'
context = build_rag_context(question)
print(context)

prompt = f"Use only this context to answer.\n\n{context}\n\nQuestion: {question}"
print('\nPrompt sent to model:\n')
print(prompt)

## LLM Evals: Accuracy, Latency, Cost

In [ ]:
golden_set = [
    {'id': 1, 'question': 'Why use RAG?', 'must_include': ['ground', 'hallucination']},
    {'id': 2, 'question': 'When use SQL retrieval?', 'must_include': ['exact', 'structured']}
]

model_outputs = [
    {'id': 1, 'answer': 'RAG grounds responses and reduces hallucination.', 'latency_ms': 820, 'cost_usd': 0.004},
    {'id': 2, 'answer': 'SQL is best for exact filters on structured data.', 'latency_ms': 640, 'cost_usd': 0.003}
]

def keyword_accuracy(golden, outputs):
    output_by_id = {o['id']: o for o in outputs}
    hits = 0
    for g in golden:
        ans = output_by_id[g['id']]['answer'].lower()
        if all(k in ans for k in g['must_include']):
            hits += 1
    return hits / len(golden)

acc = keyword_accuracy(golden_set, model_outputs)
latency = statistics.mean([o['latency_ms'] for o in model_outputs])
cost = sum([o['cost_usd'] for o in model_outputs])
print({'accuracy': round(acc, 2), 'avg_latency_ms': round(latency, 1), 'total_cost_usd': round(cost, 4)})

In [ ]:
judge_rubric = {
    'factual_grounding': 'Does the answer rely only on provided context?',
    'completeness': 'Does it address all parts of the question?',
    'clarity': 'Is it concise and easy to understand?'
}
print('Rubric for LLM-as-judge style evaluation:')
print(json.dumps(judge_rubric, indent=2))

# For live judging, call a strong model with this rubric and require JSON output.

In [ ]:
# Golden set artifact that can be versioned in git.
golden_path = 'golden_set.jsonl'
with open(golden_path, 'w', encoding='utf-8') as f:
    for row in golden_set:
        f.write(json.dumps(row) + '\n')
print('Wrote', golden_path)

In [ ]:
feedback_events = [
    {'id': 101, 'liked': True, 'reason': 'clear'},
    {'id': 102, 'liked': False, 'reason': 'hallucinated'},
    {'id': 103, 'liked': False, 'reason': 'too long'},
    {'id': 104, 'liked': True, 'reason': 'helpful'}
]

like_rate = sum(1 for e in feedback_events if e['liked']) / len(feedback_events)
failure_reasons = Counter(e['reason'] for e in feedback_events if not e['liked'])
print('Like rate:', round(like_rate, 2))
print('Top failure reasons:', dict(failure_reasons))

## LLM Agents: Simple SQL Agent with Safety

In [ ]:
def safe_sql_agent(user_request: str):
    lower = user_request.lower()

    # Basic guardrails for demo: reject risky patterns.
    blocked = ['drop', 'delete', ';', '--']
    if any(tok in lower for tok in blocked):
        return {'status': 'blocked', 'reason': 'Potentially unsafe SQL detected.'}

    if '2025q4' in lower:
        rows = retrieve_sql('2025Q4')
        return {'status': 'ok', 'data': rows}

    return {'status': 'needs_clarification', 'message': 'Specify a quarter like 2025Q4.'}

print(safe_sql_agent('Show policy updates for 2025Q4'))
print(safe_sql_agent('DROP TABLE policy_updates;'))

## MCP (Model Context Protocol) Example

In [ ]:
mcp_tool_descriptor = {
    'name': 'run_sql',
    'description': 'Execute read-only SQL against policy database',
    'input_schema': {
        'type': 'object',
        'properties': {'query': {'type': 'string'}},
        'required': ['query']
    }
}

print(json.dumps(mcp_tool_descriptor, indent=2))

## Looking Ahead

- Move from prompt-only changes to system-level iteration (retrieval, tools, eval loops).\n
- Keep golden sets current with production misses.\n
- Treat agent safety and observability as first-class requirements.